# Leave-k-out validation (Phase A)

Quantitative comparison of the Phase 2 PL Ising model against three baselines (marginal usage, co-occurrence weight, PPMI weight) on a tournament-strided held-out test set. For each test team and each $k \in \{1, 2, 3\}$, hold out $k$ random mons, score every candidate completion under each model, and measure top-K hit rate + mean reciprocal rank.

**Train/test split**: tournaments at indices $[0, 10, 20, \ldots]$ (every 10th, newest-first) form the test set; remaining 90% form the train set. Strided split keeps both halves spanning the full corpus time range — decouples "model is bad" from "meta has shifted."

**Expected ordering** if everything works as theorized: Ising > PPMI > co-occurrence > marginal. The headline comparison is **Ising vs PPMI** — that tests whether structural inference (Ising's $J$ marginalizes out indirect couplings) adds value over direct co-occurrence (PMI uses raw counts).

**Methodology notes**:
- Vocab is built on the **full corpus** for index consistency between train and test (test mons need vocab indices to be ranked). Cutoff matches `PHASE2_MIN_TEAM_COUNT = 5`.
- Model parameters $(J, h)$ are fit on **train only** — no test-set leakage in the fit.
- Model marginals at inference use mean-field iteration. Ranking-preserving without the team-size constraint, since the constraint adds a global Lagrange multiplier identical across candidates; fast (sub-second per test instance, vs minutes for full PT).

In [ ]:
from __future__ import annotations
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

import helpers
import limitless_ingest

tournaments = limitless_ingest.ingest()
print(f"tournaments:           {len(tournaments)}")
print(f"total team-appearances: {sum(len(t.teams) for t in tournaments)}")

## Train/test split

Tournaments are already in date order (newest-first) from `limitless_ingest`. Strided 90/10: test indices = `[0, 10, 20, ...]`; train = the rest. Disjoint at the tournament level, so no team in train appears in test.

In [ ]:
TEST_STRIDE = 10

test_tournaments = tournaments[::TEST_STRIDE]
train_tournaments = [t for i, t in enumerate(tournaments) if i % TEST_STRIDE != 0]

train_teams = limitless_ingest.all_teams(train_tournaments)
test_teams = limitless_ingest.all_teams(test_tournaments)
n_total = len(train_teams) + len(test_teams)

print(f"train tournaments: {len(train_tournaments):>3}   teams: {len(train_teams):>5}   ({len(train_teams)/n_total:.1%})")
print(f"test tournaments:  {len(test_tournaments):>3}   teams: {len(test_teams):>5}   ({len(test_teams)/n_total:.1%})")

## Train-only feature matrices

Vocab from the full corpus; marginal $m$, co-occurrence $C$, and PPMI built from train teams only.

In [ ]:
MIN_TEAM_COUNT = 5

# Vocab on full corpus for index consistency between train and test
species_counts = Counter(name for team in (train_teams + test_teams) for name in team)
vocab = sorted(name for name, c in species_counts.items() if c >= MIN_TEAM_COUNT)
name_to_idx = {name: i for i, name in enumerate(vocab)}
V = len(vocab)
print(f"vocab size: {V} (>= {MIN_TEAM_COUNT} appearances in full corpus)")


def teams_to_matrix(teams, name_to_idx, V):
    X = np.zeros((len(teams), V), dtype=np.int8)
    for ti, team in enumerate(teams):
        for name in team:
            j = name_to_idx.get(name)
            if j is not None:
                X[ti, j] = 1
    return X


X_train = teams_to_matrix(train_teams, name_to_idx, V)
X_test = teams_to_matrix(test_teams, name_to_idx, V)

# Marginal usage (train only)
m_train = X_train.mean(axis=0)

# Co-occurrence counts (train only): C[i, j] = # train teams containing both i and j
C_train = (X_train.astype(np.int64).T @ X_train.astype(np.int64)).astype(np.float64)
np.fill_diagonal(C_train, 0.0)

# PPMI (train only) -- reuse the helper, which expects a co-occurrence matrix
PPMI_train = helpers.build_ppmi(C_train)

print(f"m_train range:    [{m_train.min():.4f}, {m_train.max():.4f}]")
print(f"C_train off-diag: max = {int(C_train.max())}, nnz = {int((C_train > 0).sum())}")

## Pseudo-likelihood Ising fit on train data

Same per-spin logistic regression machinery as `inverse_ising_phase2.ipynb` and `app.py:load_model_phase2`, run on the train-team matrix only. L2 with `C = 0.1` to match the app's current default.

In [ ]:
PHASE2_LR_C = 0.1

J_asym = np.zeros((V, V), dtype=np.float64)
h = np.zeros(V, dtype=np.float64)
X_train_int = X_train.astype(np.int32)

for i in range(V):
    y = X_train_int[:, i]
    if y.sum() < 2 or (1 - y).sum() < 2:
        continue
    mask = np.ones(V, dtype=bool)
    mask[i] = False
    lr = LogisticRegression(penalty="l2", C=PHASE2_LR_C, solver="lbfgs", max_iter=1000)
    lr.fit(X_train_int[:, mask], y)
    h[i] = lr.intercept_[0]
    J_asym[i, mask] = lr.coef_[0]
    if (i + 1) % 25 == 0:
        print(f"  fit {i + 1} / {V}")

J = 0.5 * (J_asym + J_asym.T)
np.fill_diagonal(J, 0.0)

iu = np.triu_indices(V, 1)
print(f"\nJ off-diagonal range: [{J[iu].min():+.3f}, {J[iu].max():+.3f}]")
print(f"h range:              [{h.min():+.3f}, {h.max():+.3f}]")

## Candidate scoring functions

Each scorer takes the indices of held-in mons and returns a length-$V$ score array. Higher = more likely completion. Held-in mons are masked out at ranking time (already on the team).

- **Marginal**: $\text{score}_i = m_i$. Ignores held-in mons entirely. Tests "can the model do better than guessing the most popular mons?"
- **Co-occurrence**: $\text{score}_i = \sum_{j \in \text{held-in}} C_{ij}$. Raw co-occurrence weight summed over held-in.
- **PPMI**: $\text{score}_i = \sum_{j \in \text{held-in}} \text{PPMI}_{ij}$. Normalized co-occurrence.
- **Ising (mean-field)**: iterate $m_i \leftarrow \sigma(h_i + \sum_j J_{ij}\, m_j)$ with held-in $m_j = 1$ clamped; rank by converged $m_i$. Damped fixed-point with `damp=0.5` for stability against strong $-J$ couplings.

In [ ]:
def score_marginal(held_in_idx, *, m):
    return m.copy()


def score_cooccurrence(held_in_idx, *, C):
    return C[:, list(held_in_idx)].sum(axis=1)


def score_ppmi(held_in_idx, *, PPMI):
    return PPMI[:, list(held_in_idx)].sum(axis=1)


def score_ising_meanfield(held_in_idx, *, J, h, n_iters=200, tol=1e-4, damp=0.5):
    """Mean-field marginals with held-in mons clamped to 1.
    Returns length-V marginal array; held-in positions stay at 1."""
    V = len(h)
    fixed_mask = np.zeros(V, dtype=bool)
    fixed_mask[list(held_in_idx)] = True
    m = 1.0 / (1.0 + np.exp(-h))
    m[fixed_mask] = 1.0
    for _ in range(n_iters):
        m_new = 1.0 / (1.0 + np.exp(-(h + J @ m)))
        m_new[fixed_mask] = 1.0
        delta = float(np.max(np.abs(m_new[~fixed_mask] - m[~fixed_mask])))
        m = damp * m_new + (1.0 - damp) * m
        m[fixed_mask] = 1.0
        if delta < tol:
            break
    return m

## Held-out evaluation loop

For each in-vocab test team and each $k \in \{1, 2, 3\}$, sample one random held-out subset of size $k$. Score under each model, mask held-in mons out of the ranking, and record the rank of each held-out mon.

In [ ]:
SEED = 42
K_VALUES = [1, 2, 3]
TOP_K_THRESHOLDS = [1, 5, 10, 20]

rng = np.random.default_rng(SEED)

in_vocab_test_teams = [
    team for team in test_teams
    if len(team) == 6 and all(name in name_to_idx for name in team)
]
print(f"in-vocab test teams: {len(in_vocab_test_teams)} / {len(test_teams)}")

scorers = {
    "marginal":      lambda held_in: score_marginal(held_in, m=m_train),
    "co-occurrence": lambda held_in: score_cooccurrence(held_in, C=C_train),
    "PPMI":          lambda held_in: score_ppmi(held_in, PPMI=PPMI_train),
    "Ising (MF)":    lambda held_in: score_ising_meanfield(held_in, J=J, h=h),
}

# results[model_name][k] -> list of arrays (one per test team), each of length k
results = {name: {k: [] for k in K_VALUES} for name in scorers}

for team in in_vocab_test_teams:
    team_idx = np.array(sorted(name_to_idx[name] for name in team))
    for k in K_VALUES:
        held_out_pos = rng.choice(6, size=k, replace=False)
        held_out_idx = team_idx[held_out_pos]
        held_in_idx = np.delete(team_idx, held_out_pos)

        for name, scorer in scorers.items():
            scores = scorer(held_in_idx).astype(np.float64).copy()
            scores[held_in_idx] = -np.inf  # held-in already on team
            order = np.argsort(-scores)  # descending: rank 1 = highest score
            rank_of = np.empty(V, dtype=np.int64)
            rank_of[order] = np.arange(1, V + 1)
            results[name][k].append(rank_of[held_out_idx])

for name in scorers:
    for k in K_VALUES:
        results[name][k] = np.array(results[name][k])  # shape (n_test, k)

print(f"\nevaluated {len(in_vocab_test_teams)} teams x {len(K_VALUES)} k values x {len(scorers)} models")

## Metrics table

Top-K hit rate at $K \in \{1, 5, 10, 20\}$ and mean reciprocal rank (MRR), faceted by $k$.

In [ ]:
def hit_rate_at_k(ranks, K):
    return float((ranks <= K).mean())


def mrr(ranks):
    return float((1.0 / ranks).mean())


header = f"{'model':<18} {'k':>2}  {'MRR':>6}  " + "  ".join(f"top-{K:>2}" for K in TOP_K_THRESHOLDS)
print(header)
print("-" * len(header))
for name in scorers:
    for k in K_VALUES:
        ranks = results[name][k]
        row = f"{name:<18} {k:>2}  {mrr(ranks):>6.3f}  "
        row += "  ".join(f"{hit_rate_at_k(ranks, K):>6.1%}" for K in TOP_K_THRESHOLDS)
        print(row)
    print()

## Visual comparison

Top-K hit rate as a function of $K$, one panel per $k$.

In [ ]:
K_RANGE = list(range(1, 31))

fig, axes = plt.subplots(1, len(K_VALUES), figsize=(15, 4.5), sharey=True)
for ax, k in zip(axes, K_VALUES):
    for name in scorers:
        ranks = results[name][k]
        hits = [hit_rate_at_k(ranks, K) for K in K_RANGE]
        ax.plot(K_RANGE, hits, label=name, marker=".", markersize=4)
    ax.set_xlabel("K")
    ax.set_title(f"k = {k} held-out mons")
    ax.grid(alpha=0.3)
    ax.set_xlim(1, K_RANGE[-1])
axes[0].set_ylabel("top-K hit rate")
axes[-1].legend(loc="lower right")
plt.tight_layout()
plt.show()

## Interpretation

Read the table + plot for:

- **Sanity**: marginal < co-occurrence < PPMI ≤ Ising. If this order is violated, the harness is wrong (or the model has a real problem).
- **Headline**: how does **Ising vs PPMI** look? Larger gap = structural inference is doing work. Small gap = the model is mostly recovering what direct PMI captures, and Ising's value-add is modest at this corpus size.
- **$k$-dependence**: all models should degrade as $k$ grows. The interesting question is whether Ising's advantage holds, shrinks, or grows as the prediction becomes more underdetermined.

**Decision criterion for Phase B+C**: if the Ising margin at $k=1$ is substantial (say, top-5 hit rate 5+ percentage points above PPMI), that's strong motivation to add item-pair features — the model already does something measurable that direct co-occurrence doesn't, and enriching the feature space should compound the gain. If the margin is small or absent, revisit whether item-pair features are the right next step or whether the bottleneck is elsewhere (regularization, corpus size, model class).